In [1]:
# Importing libraries

import pandas as pd
import numpy as np


In [ ]:
# Reading in proteins to extract rs_ids

prot = pd.read_csv('FinalSNPsProteins.csv')

In [ ]:
# Saving protein SNP IDs to a text file
protein_snp_ids = prot['rsid'].dropna().unique()
with open('protein_snp_ids.txt', 'w') as f:
    for snp in protein_snp_ids:
        f.write(snp + '\n')


In [ ]:

# Ran the following in the terminal
# grep -Fw -f protein_snp_ids.txt AD_SNPs.tsv > ad_filtered.tsv -> this will extract only the rsIDs that I need that align with the proteomic IDs 

In [ ]:

# Assigning column names as were not extracted in the terminal code
column_names = ['variant_id', 'p_value', 'chromosome', 'base_pair_location', 'effect_allele', 'other_allele', 'effect_allele_frequency', 'odds_ratio', 'ci_lower', 'ci_upper', 'beta', 'standard_error', 'n_cases', 'n_controls', 'het_isq', 'het_pvalue', 'variant_alternate_id']

# Reading in dataset with the column names
ad = pd.read_csv('ad_filtered.tsv', sep='\t', header=None, names= column_names) 

In [ ]:
# Assessing how many in set

len(ad)

5573

In [29]:
# checking that all rs_ids are unique

# This counts every occurrence except the first one
duplicate_count = ad['variant_id'].duplicated().sum()

print(duplicate_count)
# there is one duplicate

1


In [30]:
# Assessing duplicates

duplicate_id = ad[ad['variant_id'].duplicated()]

duplicate_id

,variant_id,p_value,chromosome,base_pair_location,effect_allele,other_allele,effect_allele_frequency,odds_ratio,ci_lower,ci_upper,beta,standard_error,n_cases,n_controls,het_isq,het_pvalue,variant_alternate_id
3498,rs45578540,0.3216,11,118607966,T,C,0.0457,1.02,0.981,1.06,0.0196,0.0198,85934,401577,0.0,0.8196,chr11:118607966:C:T


In [ ]:
# Extracting just this id

ad[ad['variant_id'] == 'rs45578540']

# Looks like there is a difference on which effect allele is being measured -> will have to look at protein dataset to check which one to keep

,variant_id,p_value,chromosome,base_pair_location,effect_allele,other_allele,effect_allele_frequency,odds_ratio,ci_lower,ci_upper,beta,standard_error,n_cases,n_controls,het_isq,het_pvalue,variant_alternate_id
3497,rs45578540,0.5626,11,118607966,C,G,0.9988,1.149,0.718,1.839,0.1390,0.2400,26798,28624,0.0,0.9407,chr11:118607966:C:G
3498,rs45578540,0.3216,11,118607966,T,C,0.0457,1.020,0.981,1.060,0.0196,0.0198,85934,401577,0.0,0.8196,chr11:118607966:C:T


In [ ]:
# Assessing the identified rsID
prot[prot['SNP'] == 'rs45578540']

# they used T -> therefore need to remove the one with C effect allele

,SNP,beta,se,effect_allele,other_allele,eaf,exposure
3785,rs45578540,-0.761866,0.019179,T,C,0.041709,TREH


In [ ]:
# Removing the rsID of interest row that has the C allele
ad = ad[~((ad['variant_id'] == 'rs45578540') & (ad['effect_allele'] == 'C'))]

In [ ]:
# Removing uneeded columns
ad_final = ad[['variant_id' , 'p_value' , 'effect_allele' , 'other_allele' , 'beta' , 'standard_error' , 'effect_allele_frequency']]

In [ ]:
# Renaming columns to match needed format for MR

ad_final = ad_final.rename(columns={ 'variant_id': 'SNP', 'standard_error': 'se', 'p_value': 'pval', 'effect_allele_frequency': 'eaf'})

In [ ]:
# Adding outcome variable- also needed for MR

ad_final['outcome'] = "Alzheimer's Disease"


In [ ]:
# Writing outcome dataset to csv
ad_final.to_csv("finalSNPsAD.csv", index=False)